# 03. Sample QC（Quality Control） and exploratory normalization

このノートブックは、count matrixができた後に、サンプル全体の関係を見る段階である。

**この段階が答える問い**

- 同じ群のreplicateが近いか。
- read depthや検出遺伝子数が極端なサンプルはないか。
- PCAや相関で明らかな外れ値がないか。

**入力**

- `results/counts/gene_counts.tsv`
- `metadata/samples.tsv`

**出力**

- `results/sample_qc/library_size.tsv`
- `results/sample_qc/pca_logcpm.tsv`
- `results/sample_qc/*.png`

**次に進む条件**

- 大きな外れ値がない、または外れ値の扱いを決めた。
- 群間差よりbatchや個体差が支配的でないかを確認した。


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
CONFIG = json.loads((PROJECT_DIR / "config" / "analysis_config.json").read_text(encoding="utf-8"))
SAMPLES = pd.read_csv(PROJECT_DIR / CONFIG["samples_path"], sep="\t")
COUNTS_PATH = PROJECT_DIR / CONFIG["differential_expression"]["count_matrix_path"]
SAMPLE_QC_DIR = PROJECT_DIR / "results" / "sample_qc"
SAMPLE_QC_DIR.mkdir(parents=True, exist_ok=True)

if not COUNTS_PATH.exists():
    raise FileNotFoundError(f"Count matrix not found: {COUNTS_PATH}. Run notebook 02 first.")

counts = pd.read_csv(COUNTS_PATH, sep="\t")
counts = counts.set_index("gene_id")
counts = counts[SAMPLES["sample_id"]]
counts.head()


## library sizeと検出遺伝子数

library sizeは、各サンプルの総countである。検出遺伝子数は、countが1以上あるgene数である。


In [ ]:
sample_qc = pd.DataFrame(
    {
        "sample_id": counts.columns,
        "library_size": counts.sum(axis=0).values,
        "detected_genes": (counts > 0).sum(axis=0).values,
    }
).merge(SAMPLES[["sample_id", "condition", "replicate"]], on="sample_id", how="left")

sample_qc.to_csv(SAMPLE_QC_DIR / "library_size.tsv", sep="\t", index=False)
sample_qc


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=sample_qc, x="sample_id", y="library_size", hue="condition", ax=axes[0])
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_title("Library size")
sns.barplot(data=sample_qc, x="sample_id", y="detected_genes", hue="condition", ax=axes[1])
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("Detected genes")
plt.tight_layout()
plt.savefig(SAMPLE_QC_DIR / "library_size_detected_genes.png", dpi=160)
plt.show()


## logCPMを作る

PCAや相関を見るために、countを `log2(CPM + 1)` に変換する。

CPMは counts per million である。

`CPM_gs = count_gs / library_size_s * 1,000,000`

これは探索的可視化用である。DESeq2の統計検定そのものは次のノートブックで行う。


In [ ]:
library_sizes = counts.sum(axis=0)
cpm = counts.divide(library_sizes, axis=1) * 1_000_000
log_cpm = np.log2(cpm + 1)
log_cpm.head()


## PCA（Principal Component Analysis）

PCAは、サンプル間の大きな違いを2次元に圧縮して見る方法である。同じ群のreplicateが近くに集まるかを見る。


In [ ]:
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(log_cpm.T)
pca_df = pd.DataFrame(pca_scores, columns=["PC1", "PC2"])
pca_df["sample_id"] = log_cpm.columns
pca_df = pca_df.merge(SAMPLES[["sample_id", "condition", "replicate"]], on="sample_id", how="left")
pca_df.to_csv(SAMPLE_QC_DIR / "pca_logcpm.tsv", sep="\t", index=False)

explained = pca.explained_variance_ratio_ * 100
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="condition", style="replicate", s=90, ax=ax)
for _, row in pca_df.iterrows():
    ax.text(row["PC1"], row["PC2"], row["sample_id"], fontsize=8, ha="left", va="bottom")
ax.set_xlabel(f"PC1 ({explained[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({explained[1]:.1f}%)")
ax.set_title("PCA on logCPM")
plt.tight_layout()
plt.savefig(SAMPLE_QC_DIR / "pca_logcpm.png", dpi=160)
plt.show()


## sample correlation heatmap

サンプル間相関は、サンプル同士が全体としてどれくらい似ているかを見る。同じ群のreplicateで高い相関が期待される。


In [ ]:
correlation = log_cpm.corr(method="pearson")
correlation.to_csv(SAMPLE_QC_DIR / "sample_correlation_logcpm.tsv", sep="\t")

plt.figure(figsize=(7, 6))
sns.heatmap(correlation, cmap="vlag", center=0.95, annot=True, fmt=".2f", square=True)
plt.title("Sample correlation on logCPM")
plt.tight_layout()
plt.savefig(SAMPLE_QC_DIR / "sample_correlation_logcpm.png", dpi=160)
plt.show()


## 高発現遺伝子を見る

高発現遺伝子はライブラリの構成に強く影響する。ミトコンドリア遺伝子、rRNA、ストレス応答遺伝子などが上位に出ることがある。


In [ ]:
mean_counts = counts.mean(axis=1).sort_values(ascending=False)
top_genes = mean_counts.head(30).rename("mean_count").reset_index()
top_genes.to_csv(SAMPLE_QC_DIR / "top_mean_count_genes.tsv", sep="\t", index=False)
top_genes.head(20)


**よくある間違い**

- count matrixができた直後に、PCAや相関を見ずにDESeq2へ進む。
- 外れ値らしきサンプルを見つけても、理由を調べずに機械的に削除する。

**小さい練習**

PCAで一番離れているサンプルがある場合、そのサンプルの `library_size` と `detected_genes` を見比べよう。
